In [10]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams
import numpy as np

load_dotenv()
QDRANT_API_KEY = os.getenv("API_KEY")
QDRANT_URL = os.getenv("URL")



In [11]:
client = QdrantClient(
    url= QDRANT_URL, 
    api_key= QDRANT_API_KEY,
)


client.recreate_collection(): <br>
client.upsert()

                                  client.get_collections()

methods: <br>
-client.get_collections(): returns a Pydantic model collectionResponse which has one attribute collections<br><br>
-client.get_collections() has a lot of methods like model_dump() which transforms the Pydantic model collectionResponse to a dictionnary of key 'collections' so we can say that 
client.get_collections().model_dump()['collections'] and client.get_collections().collections are the same thing
<br><br>
-client.get_collection().model_dump_json(indent=2) transform the collections into a json file

In [12]:
collections = client.get_collections()
print(len(collections.collections))

1


In [13]:
print(collections.model_dump_json(indent=2))

{
  "collections": [
    {
      "name": "fitness_docs"
    }
  ]
}


In [14]:
CLS = collections.model_dump()["collections"]

for collection in CLS:
    print(collection['name'])

CLS = collections.collections


for collection in CLS:
    print(collection.name)

fitness_docs
fitness_docs


In [15]:
print(collections.model_dump())


{'collections': [{'name': 'fitness_docs'}]}


In [16]:
A = np.zeros((100, 7))
a = np.random.random((100,7))


                        client.recreate_collection(collection_name, vectors_config)

This is what we use to create a new_collection, in collection_name we put the name of our collection and in vectors_config we specify the size and you choose the distance you want

In [17]:
client.recreate_collection(
    collection_name = "fit_database" ,
    vectors_config = VectorParams(size = 200, distance=Distance.DOT)
                           )

C:\Users\adamh\AppData\Local\Temp\ipykernel_10356\3534900326.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

                                    client.upsert() and client.delete()

In [24]:
client.upsert(
    collection_name="fitness_docs",
    points=[
        models.PointStruct(
            id=1,
            vector=[1.0, 2.0, 3.0],
            payload={"author": "Arnold", "text": "Train legs twice a week"}
        )
    ]
)


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [25]:
client.delete(
    collection_name="fitness_docs",
    points_selector=models.PointIdsList(points=[1, 2, 3])
)


UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
client.delete(
    collection_name="fitness_docs",
    points_selector=models.FilterSelector(
        filter=Filter( must=[FieldCondition(key="author", match=MatchValue(value="Arnold"))] )
    )
)


                                  client.search

In [ ]:
client.search(collection_name="fitness_docs", query_vector=embedding, limit=5)

                                    Models.PointStruct(id, vector, playload)

-models.PointStruct(id, vector, playload) 

                                collection_name: "database"

In [2]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


In [32]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams
import numpy as np
from app.services.openai_client import get_embedding

load_dotenv()
QDRANT_API_KEY = os.getenv("API_KEY")
QDRANT_URL = os.getenv("URL")


ModuleNotFoundError: No module named 'app'

In [4]:
import requests

headers = {"Authorization": f"Bearer {os.getenv('API_KEY')}"}
url = "https://270d1d27-de00-4512-aff8-fc2cee181307.europe-west3-0.gcp.cloud.qdrant.io/collections"
response = requests.get(url, headers=headers)
print(response.status_code, response.text)


403 {"error":"forbidden"}


In [3]:
import os
os.chdir("/Users/adamh/Desktop/fit_bot/back-end/app")

In [5]:
import os
import json
import requests
from dotenv import load_dotenv
from services.openai_client import get_embedding  # your embedding function

# Load environment variables
load_dotenv()

QDRANT_URL = "https://270d1d27-de00-4512-aff8-fc2cee181307.europe-west3-0.gcp.cloud.qdrant.io"
API_KEY = os.getenv("API_KEY")

def query_qdrant_and_get_text(text, collection="database", limit=1):
    """
    Generate an embedding for the input text,
    query Qdrant Cloud, and print the top matching payload text.
    """
    # Step 1: Get the embedding
    embedding = get_embedding(text)

    # Step 2: Build request payload
    payload = {
        "query": embedding,
        "limit": limit,
        "with_payload": True  # 👈 this makes Qdrant return stored payload data
    }

    # Step 3: Set headers
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # Step 4: Send request
    response = requests.post(
        f"{QDRANT_URL}/collections/{collection}/points/query",
        headers=headers,
        data=json.dumps(payload),
        timeout=30
    )

    # Step 5: Parse results
    if response.status_code != 200:
        print("❌ Query failed:", response.text)
        return None

    results = response.json()["result"]["points"]
    if not results:
        print("No results found.")
        return None

    # Step 6: Extract top point info
    top_point = results[0]
    point_id = top_point["id"]
    score = top_point["score"]
    payload_data = top_point.get("payload", {})
    text_value = payload_data.get("text", "[No text field in payload]")

    print("✅ Top match found:")
    print(f"ID: {point_id}")
    print(f"Score: {score}")
    print(f"Text: {text_value}\n")

    return text_value


# Example usage
if __name__ == "__main__":
    query_text = "can you tell me which muscle groups I should prioritize to appear esthetic"
    matched_text = query_qdrant_and_get_text(query_text)


❌ Query failed: {"error":"forbidden"}


In [18]:
import os
os.chdir("/Users/adamh/Desktop/fit_bot/back-end/app")

In [31]:

from dotenv import load_dotenv
from services.openai_client import get_embedding  # your embedding function

load_dotenv()
API_KEY = os.getenv("API_KEY")


In [28]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url="https://270d1d27-de00-4512-aff8-fc2cee181307.europe-west3-0.gcp.cloud.qdrant.io", 
    api_key=API_KEY,
    port=443
)

print(qdrant_client.get_collections())

collections=[CollectionDescription(name='vector_database'), CollectionDescription(name='fitness_docs'), CollectionDescription(name='database'), CollectionDescription(name='fit_database')]


In [33]:


text = "can you tell me the best morning workout routine"
embedding = get_embedding(text)
data = qdrant_client.search(collection_name="database", query_vector=embedding, limit=1)



C:\Users\adamh\AppData\Local\Temp\ipykernel_14928\2761766768.py:3: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  data = qdrant_client.search(collection_name="database", query_vector=embedding, limit=1)


In [40]:
print(data[0].payload['text'])

cycle, give it a go. Avoid “downgrading,” however, unless you have to.
If you’re not sure which routine to follow, pick the one you know you
can stick to. If you’re sure you can get to the gym four days per week but
not five, go with the four-day routine.
Remember—consistency is key to results with any workout program,
and this is especially true when you’re using percentages of one-rep maxes
like you’ll do in this one.
All right, on to the routines!
The Five-Day Routine
Workout 1 Workout 2 Workout 3 Workout 4 Workout 5
Upper Pull & Calves Upper Legs & Calves Upper
Body A Body B Body C
If you have the time and inclination, start here for your first macrocycle.
You can always try the other routines in later macrocycles.
Most people who follow this routine train Monday through Friday and
take the weekends off, but you can incorporate your rest days however
you’d like. The important thing is you do each of the workouts every seven
days in the order given.
I recommend including at least on

In [34]:
top_point = data[0]
payload_data = top_point.get("payload", {})
text_value = payload_data.get("text", "[No text field in payload]")
print(text_value)

AttributeError: 'ScoredPoint' object has no attribute 'get'